# AraBERT Multi-Head (single model, 6 targets)

One `aubmindlab/bert-base-arabertv02` encoder with **6 heads**, predicting all targets at once.
No external LLM APIs, open-source only, fixed seed, reproducible.

| Target | Classes |
|---|---|
| Qatar Related | 0 / 1 |
| Geography | G-1..G-12, NO_GEOGRAPHY (13) |
| Politics & Conflict | NO_POLITICS, PC-1, PC-2 (3) |
| Health & Wellbeing | NO_HEALTH, H-1, H-2 (3) |
| Science | NO_SCIENCE, SC-1, SC-2, SC-3 (4) |
| Sports | NO_SPORTS, SP-1..SP-4 (5) |

**Stages**
1. **CV harness** — data, label encoders, multi-label-aware `MultilabelStratifiedKFold`, per-head
   class weights, exact Zindi scorer, per-class F1 helper, and a smoke test (no real training).
2. **Full training loop** — fold-0 `max_length` sweep (256/384/512), then full 5-fold CV at the winner,
   with per-fold and OOF per-class F1.
3. **Inference + submission** — averaged fold-model probabilities → `submission.csv`.

**Scoring** = mean over the 6 columns of per-column weighted F1 (Zindi's function, replicated exactly).


In [ ]:
%pip install -q "transformers>=4.40,<5" iterative-stratification

In [ ]:
import copy
import importlib.metadata as im
import math
import os
import random
import time
import warnings
from pathlib import Path

# must be set before CUDA is initialized (helps reproducibility of cuBLAS GEMMs)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

warnings.filterwarnings("ignore")

SEED = 42


def seed_everything(s=SEED):
    random.seed(s)
    os.environ["PYTHONHASHSEED"] = str(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)


seed_everything()
if torch.cuda.is_available():
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception as e:
        print("deterministic algorithms not enabled:", type(e).__name__)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print("device:", DEVICE, "| torch", torch.__version__, "| transformers", im.version("transformers"))

CANDIDATE_DIRS = [
    Path("/content"),
    Path.cwd(),
    Path("/Users/ahadhussain/Desktop/Scratch projects/googleaihackathon"),
]
DATA_DIR = next((d for d in CANDIDATE_DIRS if (d / "Train.csv").exists()), CANDIDATE_DIRS[0])
OUTPUT_DIR = (DATA_DIR if str(DATA_DIR).startswith("/content") else Path("/tmp/arabert_sub"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("DATA_DIR:", DATA_DIR, "| OUTPUT_DIR:", OUTPUT_DIR)

ID_COLUMN = "ID"
TEXT_COLUMN = "text"
TARGET_COLUMNS = [
    "Qatar Related",
    "Geography",
    "Politics & Conflict",
    "Health & Wellbeing",
    "Science",
    "Sports",
]

# ----------------------------- config -----------------------------
MODEL_NAME = "aubmindlab/bert-base-arabertv02"
MAX_LEN_LIST = [256, 384, 512]
N_FOLDS = 5
EPOCHS = 3
BATCH_SIZE = 16
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
DROPOUT = 0.1
MAX_CLASS_WEIGHT = 20.0     # clip inverse-frequency weights (G-1 has 2 examples)
NUM_WORKERS = 2 if DEVICE.type == "cuda" else 0
VERBOSE_PER_CLASS = True    # print per-class F1 per fold

print({k: v for k, v in dict(MODEL_NAME=MODEL_NAME, MAX_LEN_LIST=MAX_LEN_LIST, N_FOLDS=N_FOLDS,
                             EPOCHS=EPOCHS, BATCH_SIZE=BATCH_SIZE, LR=LR).items()})

In [ ]:
# ---------- exact package versions for reproducibility ----------
PKGS = ["torch", "transformers", "tokenizers", "scikit-learn", "pandas", "numpy",
        "iterative-stratification", "accelerate"]
lines = []
for p in PKGS:
    try:
        lines.append(f"{p}=={im.version(p)}")
    except Exception:
        lines.append(f"# {p} not installed")
req_text = "\n".join(lines) + "\n"
REQ_PATH = OUTPUT_DIR / "requirements.txt"
REQ_PATH.write_text(req_text)
print("requirements ->", REQ_PATH)
print(req_text)

## Stage 1 — CV harness (validate before real training)

Builds the data pipeline, the multi-label stratification, class weights and the exact scorer, then
runs a tiny smoke test to prove the wiring works before any full training.

In [ ]:
train_df = pd.read_csv(DATA_DIR / "Train.csv")
test_df = pd.read_csv(DATA_DIR / "Test.csv")
for c in TARGET_COLUMNS:
    train_df[c] = train_df[c].astype(str)
train_df[TEXT_COLUMN] = train_df[TEXT_COLUMN].fillna("")
test_df[TEXT_COLUMN] = test_df[TEXT_COLUMN].fillna("")
print("train:", train_df.shape, "| test:", test_df.shape)

label_encoders = {c: LabelEncoder().fit(train_df[c]) for c in TARGET_COLUMNS}
NUM_CLASSES = {c: len(label_encoders[c].classes_) for c in TARGET_COLUMNS}
y_all = {c: label_encoders[c].transform(train_df[c].values) for c in TARGET_COLUMNS}
for c in TARGET_COLUMNS:
    vc = train_df[c].value_counts()
    print(f"\n{c} ({NUM_CLASSES[c]} classes)")
    print(vc.to_frame("count").T.to_string())

In [ ]:
# ---------- multi-label-aware stratification ----------
# one-hot every target and concatenate -> (n_samples, 30) binary indicator matrix
ind = np.hstack([
    np.eye(NUM_CLASSES[c])[y_all[c]] for c in TARGET_COLUMNS
]).astype(np.int8)
print("indicator matrix:", ind.shape)

mskf = MultilabelStratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = list(mskf.split(train_df, ind))
for i, (tr, va) in enumerate(folds):
    print(f"fold {i}: train {len(tr)} val {len(va)}")

# rare-class coverage check
RARE = {"Geography": ["G-1", "G-10"], "Science": ["SC-2"], "Sports": ["SP-1"]}
for c, classes in RARE.items():
    presence = []
    for cls in classes:
        y = y_all[c]
        cnt = [int((y[va] == list(label_encoders[c].classes_).index(cls)).sum()) for _, va in folds]
        presence.append(f"{cls}: val counts/fold {cnt}")
    print(f"rare coverage {c}: " + " | ".join(presence))

In [ ]:
# ---------- per-head class weights (inverse frequency, clipped) ----------
def class_weights_for(y, num_classes, max_w=MAX_CLASS_WEIGHT):
    cnt = np.bincount(y, minlength=num_classes).astype(np.float64)
    w = np.ones(num_classes, dtype=np.float64)
    nz = cnt > 0
    w[nz] = len(y) / (num_classes * cnt[nz])
    return np.clip(w, 1.0, max_w)


print("class weights per head (full train):")
for c in TARGET_COLUMNS:
    w = class_weights_for(y_all[c], NUM_CLASSES[c])
    print(f"  {c:20s} " + ", ".join(f"{cls}={w[i]:.2f}" for i, cls in enumerate(label_encoders[c].classes_)))


# ---------- exact Zindi scorer ----------
def zindi_score(actual, predicted, columns=TARGET_COLUMNS):
    scores = {c: f1_score(actual[c], predicted[c], average="weighted") for c in columns}
    scores["mean"] = float(np.mean(list(scores.values())))
    return scores


def compact_report(y_true, y_pred, classes):
    rep = classification_report(y_true, y_pred, labels=classes, output_dict=True, zero_division=0)
    return {c: (round(rep[c]["f1-score"], 3), int(rep[c]["support"])) for c in classes}


print("\nzindi_score + compact_report ready")

In [ ]:
# ---------- model / dataset / collate ----------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
PAD_ID = tokenizer.pad_token_id


class NewsDataset(Dataset):
    def __init__(self, texts, max_len, labels=None):
        self.enc = tokenizer(list(texts), truncation=True, max_length=max_len)
        self.labels = labels

    def __len__(self):
        return len(self.enc["input_ids"])

    def __getitem__(self, i):
        item = {
            "input_ids": torch.tensor(self.enc["input_ids"][i], dtype=torch.long),
            "attention_mask": torch.tensor(self.enc["attention_mask"][i], dtype=torch.long),
        }
        if self.labels is not None:
            item["labels"] = torch.tensor([self.labels[c][i] for c in TARGET_COLUMNS], dtype=torch.long)
        return item


def collate(features):
    maxlen = max(len(f["input_ids"]) for f in features)
    ids = torch.full((len(features), maxlen), PAD_ID, dtype=torch.long)
    attn = torch.zeros((len(features), maxlen), dtype=torch.long)
    for i, f in enumerate(features):
        L = len(f["input_ids"])
        ids[i, :L] = f["input_ids"]
        attn[i, :L] = 1
    batch = {"input_ids": ids, "attention_mask": attn}
    if "labels" in features[0]:
        batch["labels"] = torch.stack([f["labels"] for f in features])
    return batch


class MultiHeadAraBERT(nn.Module):
    def __init__(self, model_name, num_classes, dropout=DROPOUT):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        h = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.heads = nn.ModuleDict({c: nn.Linear(h, num_classes[c]) for c in TARGET_COLUMNS})

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0]
        out = self.dropout(out)
        return {c: self.heads[c](out) for c in TARGET_COLUMNS}


@torch.no_grad()
def predict(model, loader):
    model.eval()
    out = {c: [] for c in TARGET_COLUMNS}
    for batch in loader:
        ids = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        logits = model(ids, mask)
        for c in TARGET_COLUMNS:
            out[c].append(torch.softmax(logits[c].float(), 1).cpu().numpy())
    return {c: np.concatenate(v, 0) for c, v in out.items()}


# ---------- smoke test: tiny subset, few steps, no real training ----------
seed_everything()
tr_idx = folds[0][0][:64]
va_idx = folds[0][1][:32]
smoke_tr = NewsDataset(train_df[TEXT_COLUMN].iloc[tr_idx].tolist(), 128,
                       {c: y_all[c][tr_idx] for c in TARGET_COLUMNS})
smoke_dl = DataLoader(smoke_tr, batch_size=8, shuffle=True, collate_fn=collate)
smoke_model = MultiHeadAraBERT(MODEL_NAME, NUM_CLASSES).to(DEVICE)
cw = {c: torch.tensor(class_weights_for(y_all[c][tr_idx], NUM_CLASSES[c]), dtype=torch.float32).to(DEVICE)
      for c in TARGET_COLUMNS}
opt = torch.optim.AdamW(smoke_model.parameters(), lr=LR)
smoke_model.train()
for step, batch in enumerate(smoke_dl):
    ids = batch["input_ids"].to(DEVICE)
    mask = batch["attention_mask"].to(DEVICE)
    labels = batch["labels"].to(DEVICE)
    opt.zero_grad()
    logits = smoke_model(ids, mask)
    loss = sum(F.cross_entropy(logits[c], labels[:, i], weight=cw[c])
               for i, c in enumerate(TARGET_COLUMNS)) / len(TARGET_COLUMNS)
    loss.backward()
    opt.step()
    print(f"smoke step {step}: loss={loss.item():.4f} | logits shapes " +
          ", ".join(f"{c.split()[0]}={tuple(logits[c].shape)}" for c in TARGET_COLUMNS))
    if step >= 2:
        break
print("STAGE 1 SMOKE TEST PASSED")
del smoke_model
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

## Stage 2 — full training loop

First a fold-0 `max_length` sweep (256/384/512), then full 5-fold CV at the winning length.
Class weights are recomputed per fold from that fold's training split (no leakage).

In [ ]:
def fold_class_weights(tr_idx):
    return {c: torch.tensor(class_weights_for(y_all[c][tr_idx], NUM_CLASSES[c]), dtype=torch.float32).to(DEVICE)
            for c in TARGET_COLUMNS}


def train_one(tr_idx, va_idx, max_len, epochs=EPOCHS, verbose=True):
    seed_everything()
    ds_tr = NewsDataset(train_df[TEXT_COLUMN].iloc[tr_idx].tolist(), max_len,
                        {c: y_all[c][tr_idx] for c in TARGET_COLUMNS})
    ds_va = NewsDataset(train_df[TEXT_COLUMN].iloc[va_idx].tolist(), max_len,
                        {c: y_all[c][va_idx] for c in TARGET_COLUMNS})
    dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate,
                       num_workers=NUM_WORKERS, pin_memory=True)
    dl_va = DataLoader(ds_va, batch_size=BATCH_SIZE * 2, shuffle=False, collate_fn=collate,
                       num_workers=NUM_WORKERS)
    model = MultiHeadAraBERT(MODEL_NAME, NUM_CLASSES).to(DEVICE)
    cw = fold_class_weights(tr_idx)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    total = len(dl_tr) * epochs
    sched = get_linear_schedule_with_warmup(opt, int(total * WARMUP_RATIO), total)
    use_amp = DEVICE.type == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    best = (-1.0, None, None)
    for ep in range(epochs):
        model.train()
        running = 0.0
        for batch in dl_tr:
            ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            opt.zero_grad()
            with torch.cuda.amp.autocast(enabled=use_amp):
                logits = model(ids, mask)
                loss = sum(F.cross_entropy(logits[c], labels[:, i], weight=cw[c])
                           for i, c in enumerate(TARGET_COLUMNS)) / len(TARGET_COLUMNS)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            running += loss.item()

        va_p = predict(model, dl_va)
        preds = {c: label_encoders[c].inverse_transform(va_p[c].argmax(1)) for c in TARGET_COLUMNS}
        sc = zindi_score(train_df.iloc[va_idx].reset_index(drop=True), pd.DataFrame(preds))
        if verbose:
            print(f"    ep {ep + 1}: loss={running / len(dl_tr):.4f} zindi={sc['mean']:.4f}")
        if sc["mean"] > best[0]:
            best = (sc["mean"], copy.deepcopy(model.state_dict()), va_p)
    model.load_state_dict(best[1])
    return model, best[2], best[0]

In [ ]:
# ---------- fold-0 max_length sweep ----------
sweep = {}
tr0, va0 = folds[0]
for ml in MAX_LEN_LIST:
    print(f"=== max_len {ml} (fold 0) ===")
    t0 = time.time()
    model, va_p, score = train_one(tr0, va0, ml)
    preds = {c: label_encoders[c].inverse_transform(va_p[c].argmax(1)) for c in TARGET_COLUMNS}
    sc = zindi_score(train_df.iloc[va0].reset_index(drop=True), pd.DataFrame(preds))
    sweep[ml] = sc
    print(f"  mean={sc['mean']:.4f} | " + " ".join(f"{c.split()[0]}={sc[c]:.3f}" for c in TARGET_COLUMNS)
          + f" | {time.time() - t0:.0f}s")
    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

BEST_MAX_LEN = max(sweep, key=lambda k: sweep[k]["mean"])
print(f"\nbest max_len = {BEST_MAX_LEN} (mean {sweep[BEST_MAX_LEN]['mean']:.4f})")

# token length percentiles at the best length
lengths = [len(x) for x in tokenizer(train_df[TEXT_COLUMN].tolist(), truncation=False)["input_ids"]]
print("token length percentiles:", {p: int(np.percentile(lengths, p)) for p in [50, 75, 90, 95, 99, 100]})

In [ ]:
# ---------- full 5-fold CV at the winning length ----------
oof = {c: np.zeros((len(train_df), NUM_CLASSES[c]), dtype=np.float32) for c in TARGET_COLUMNS}
test_fold_probs = {c: [] for c in TARGET_COLUMNS}
fold_scores = []

test_ds = NewsDataset(test_df[TEXT_COLUMN].tolist(), BEST_MAX_LEN)
dl_te = DataLoader(test_ds, batch_size=BATCH_SIZE * 2, shuffle=False, collate_fn=collate,
                   num_workers=NUM_WORKERS)

for fi, (tr_idx, va_idx) in enumerate(folds):
    print(f"\n===== fold {fi} (max_len {BEST_MAX_LEN}) =====")
    model, va_p, score = train_one(tr_idx, va_idx, BEST_MAX_LEN)
    for c in TARGET_COLUMNS:
        oof[c][va_idx] = va_p[c]
    preds = {c: label_encoders[c].inverse_transform(va_p[c].argmax(1)) for c in TARGET_COLUMNS}
    va_actual = train_df.iloc[va_idx].reset_index(drop=True)
    fold_sc = zindi_score(va_actual, pd.DataFrame(preds))
    fold_scores.append(fold_sc)
    print(f"  fold {fi} zindi={fold_sc['mean']:.4f}")
    if VERBOSE_PER_CLASS:
        for c in TARGET_COLUMNS:
            rep = compact_report(train_df[c].iloc[va_idx].values, preds[c], list(label_encoders[c].classes_))
            print(f"    {c:20s} " + " ".join(f"{k}:{v[0]}({v[1]})" for k, v in rep.items()))
    tp = predict(model, dl_te)
    for c in TARGET_COLUMNS:
        test_fold_probs[c].append(tp[c])
    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

oof_preds = {c: label_encoders[c].inverse_transform(oof[c].argmax(1)) for c in TARGET_COLUMNS}
oof_sc = zindi_score(train_df, pd.DataFrame(oof_preds))
print("\n=== OOF per-column weighted F1 ===")
for c in TARGET_COLUMNS:
    print(f"  {c:20s} {oof_sc[c]:.4f}")
print(f"  {'MEAN':20s} {oof_sc['mean']:.4f}")
print(f"per-fold means: {[round(f['mean'], 4) for f in fold_scores]}")
print("\n=== OOF per-class F1 (score, support) ===")
for c in TARGET_COLUMNS:
    rep = compact_report(train_df[c].values, oof_preds[c], list(label_encoders[c].classes_))
    print(f"  {c:20s} " + " ".join(f"{k}:{v[0]}({v[1]})" for k, v in rep.items()))

test_probs = {c: np.mean(test_fold_probs[c], axis=0) for c in TARGET_COLUMNS}

## Stage 3 — inference + submission

Averages the probabilities of the 5 fold models and writes `submission.csv` in the exact
`SampleSubmission.csv` column order, with IDs from `Test.csv`.

In [ ]:
sub_pred = {c: label_encoders[c].inverse_transform(test_probs[c].argmax(1)) for c in TARGET_COLUMNS}
submission = test_df[[ID_COLUMN]].copy()
for c in TARGET_COLUMNS:
    submission[c] = sub_pred[c]
submission = submission[[ID_COLUMN] + TARGET_COLUMNS]

assert len(submission) == len(test_df), "row count mismatch"
assert set(submission[ID_COLUMN]) == set(test_df[ID_COLUMN]), "ID mismatch"
assert list(submission.columns) == [ID_COLUMN] + TARGET_COLUMNS

FINAL_PATH = OUTPUT_DIR / "submission.csv"
submission.to_csv(FINAL_PATH, index=False)
print("Saved:", FINAL_PATH)
display(submission.head())

In [ ]:
print("model:", MODEL_NAME, "| best max_len:", BEST_MAX_LEN)
print("OOF zindi (mean weighted F1 over 6 columns):", round(oof_sc["mean"], 4))
print("requirements.txt written to:", REQ_PATH.resolve())
print("submission:", FINAL_PATH)

In [ ]:
# optional: download from Colab
# from google.colab import files
# files.download(str(FINAL_PATH))